In [20]:
from datasets import load_dataset
import torch
import torch.nn as nn
import numpy as np
import polars as pl
import io
from PIL import Image
from torchvision.transforms import v2

In [2]:
ds = load_dataset("dragonintelligence/CIFAKE-image-dataset")

## Fake 0, Real 1

In [85]:
df_train=ds['train'].to_polars()
df_test=ds['test'].to_polars()

### converting the images into pytorch suitable format --requires more memory

In [86]:
transform=v2.Compose([
      v2.RandomHorizontalFlip(p=0.4),
      v2.ToDtype(torch.float16, scale=True),
      v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [87]:
##TODO prepare the dataset and put all transformations there

### Prepare the class and the data loaders to start modelling

In [90]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
# specify a method to convert the images into a tensor
# available once: 1. through numpy, 2. through torchvision
class dataset(Dataset):
    
    def __init__(self, data, transform=None):
        super(dataset, self).__init__()
        self.polars_data=data
        self.transform=transform
        self.pil_images=None
        self.tensor=None
        self.tensor_np=None
        self.classes=None
        self.convert_images_into_bytes()
            
    def __len__(self):
        if self.tensor!=None:
            return self.tensor.shape[0]
        return None
            
    def __getitem__(self, index):
        if self.tensor!=None:
            return self.tensor[0]
        return None        
    def convert_images_into_bytes(self):
        self.polars_data=self.polars_data.with_columns(
            pl.col('image').struct.field('path').alias("image_path"),
            pl.col('image').struct.field('bytes').alias('image_bytes')
        )
        self.pil_images=[Image.open(io.BytesIO(img)) for img in self.polars_data['image_bytes'].to_list()]
    
        return self

    def transform_to_tensor(self): # d: a list of the PIL images
        images=[v2.functional.to_image(img) for img in self.pil_images]
        images=torch.stack(images)
        self.tensor=self.transform(images)
        return self

    def transform_to_tensor_through_numpy(self):
        array=np.array(self.pil_images)
        self.tensor_np=torch.from_numpy(array)    
        return self

    def fill_classes(self):
        self.classes= self.polars_data.select(pl.col('label')).to_torch()
        return self

    def prepare(self):
        self.transform_to_tensor()
        self.fill_classes()
        return self.tensor, self.classes

    def loading(self):
        image, labels=self.prepare()
        return train_test_split(image, label, random_state=42, test_size=0.3)

In [91]:
train_obj=dataset(df_train, transform)
test_obj=dataset(df_test, transform)

In [ ]:
df_train, class_train=train_obj.loading()
df_eval, class_eval=test_obj.loading()

In [75]:

class ConvNN(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, padding=1, stride=1, num_classes=2, device):
        super(ConvNN, self).__init__()
        self.modelling=nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel, padding=padding, stride=stride, device=device),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(out_ch, 32, kernel_size=kernel+2, padding=padding, stride=stride, device=device),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(32, 64, kernel_size=kernel +4, padding=padding, stride=stride, device=device),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=kernel+2, padding=padding, stride=stride, device=device),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(128, 256, kernel_size=kernel, padding=padding, stride=stride, device=device),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.output=nn.Sequential(
            nn.Conv2d(256, num_class, kernel_size=3),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )

    def forward(self, x):
        x=self.modelling(x)

        return self.output(x)

In [76]:
from torch.utils.data import DataLoader
def data_loading(data):
    return DataLoader(data, batch_size=50, shuffle=False, num_workers=1)

In [78]:
# train one epoch

def train_one_epoch(image_loader, class_loader, model, device, opt):
    train_loss=0
    criterion=nn.CrossEntropyLoss()
    model.train()
    for images, label in zip(image_loader, class_loader):
        images, label=images.to(device, non_blocking=True), label.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        output=model(images)
        loss=criterion(output, label)
        loss.backward()
        opt.step()

        train_loss+=loss.item()
        
    return train_loss

In [80]:
# evaluate one epoch

def evaluate_one_epoch(image_loader, class_loader, model, device, opt):
    eval_loss=0
    criterion=nn.CrossEntropyLoss()
    model.eval()
    with torch.no_grad():
        for images, label in zip(image_loader, class_loader):
            images, label= images.to(device, non_blocking=True), label.to(device, non_blocking=True)
            ouptut=model(images)
            loss=criterion(output, label)
            eval_loss+=loss.item()
            
    return eval_loss

In [ ]:
train_loader, train_class=data_loading(df_train), data_loading(class_train)
eval_loader, eval_class=data_loading(df_test), data_loading(class_test)